# Cập Nhật Tên Bảng Trong Qdrant DB Từ Dữ Liệu Processed Data

Notebook này thực hiện công việc:
1. Đọc danh sách file CSV trong thư mục `rag_module/ViFinQA/processed_data` (hoặc `processed_data`).
2. Trích xuất tên bảng mới (`Ten_Bang`), nguồn (`Tep_Nguon`), và số dòng (`table_line`/`source_line`).
3. Cập nhật trực tiếp thông tin tên bảng mới vào bộ sưu tập `financial_tables` trên Qdrant Local DB và file BM25 index.

In [1]:
import pickle
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import pandas as pd
from qdrant_client import QdrantClient
from tqdm import tqdm

_HERE = Path('.').resolve()
PROCESSED_DATA_DIR = _HERE / "ViFinQA" / "processed_data"
if not PROCESSED_DATA_DIR.exists():
    PROCESSED_DATA_DIR = _HERE / "processed_data"

QDRANT_DB_PATH = _HERE / "qdrant_local_db"
if not QDRANT_DB_PATH.exists():
    QDRANT_DB_PATH = _HERE / "test" / "qdrant_local_db"

BM25_PATH = _HERE / "bm25_index.pkl"
if not BM25_PATH.exists():
    BM25_PATH = _HERE / "test" / "bm25_index.pkl"

COLLECTION_NAME = "financial_tables"

print(f"Processed Data Path : {PROCESSED_DATA_DIR}")
print(f"Qdrant Local DB Path: {QDRANT_DB_PATH}")
print(f"BM25 Index Path     : {BM25_PATH}")

Processed Data Path : D:\hobby_project\cocopila\r2AI_2026\rag_module\ViFinQA\processed_data
Qdrant Local DB Path: D:\hobby_project\cocopila\r2AI_2026\rag_module\qdrant_local_db
BM25 Index Path     : D:\hobby_project\cocopila\r2AI_2026\rag_module\bm25_index.pkl


In [2]:
# Hàm hỗ trợ trích xuất số dòng và chuẩn hóa đường dẫn
def extract_line_number(tep_nguon: str, csv_path: Path) -> int:
    m = re.search(r"@line_(\d+)", tep_nguon)
    if m:
        return int(m.group(1))
    m_file = re.search(r"@line_(\d+)", csv_path.name)
    if m_file:
        return int(m_file.group(1))
    return 1

def normalize_rel_path(p_str: str) -> str:
    p_str = p_str.replace("\\", "/")
    if "processed_data/" in p_str:
        return p_str.split("processed_data/")[-1]
    return Path(p_str).name

In [3]:
# Đọc thông tin tên bảng từ các file CSV trong processed_data
csv_files = list(PROCESSED_DATA_DIR.rglob("*.csv"))
print(f"Tìm thấy {len(csv_files)} file CSV trong {PROCESSED_DATA_DIR}")

csv_meta_by_relpath: Dict[str, Dict[str, Any]] = {}
csv_meta_by_base_tep: Dict[str, Dict[str, Any]] = {}

for csv_file in tqdm(csv_files, desc="Reading CSV metadata"):
    try:
        df = pd.read_csv(csv_file, nrows=1, dtype=str)
        if df.empty:
            continue
        row = df.iloc[0]
        ten_bang = str(row.get("Ten_Bang", "")).strip()
        tep_nguon = str(row.get("Tep_Nguon", "")).strip()
        line_num = extract_line_number(tep_nguon, csv_file)
        rel_path = normalize_rel_path(csv_file.as_posix())
        base_tep = re.sub(r"@line_\d+", "", tep_nguon).strip()
        meta_info = {
            "Ten_Bang": ten_bang,
            "Tep_Nguon": tep_nguon,
            "table_line": line_num,
            "source_line": line_num,
            "csv_path": csv_file.as_posix(),
        }
        csv_meta_by_relpath[rel_path] = meta_info
        if base_tep:
            csv_meta_by_base_tep[base_tep] = meta_info
    except Exception as e:
        print(f"Warning reading {csv_file}: {e}")

print(f"Đã nạp metadata từ {len(csv_meta_by_relpath)} đường dẫn CSV.")

Tìm thấy 163128 file CSV trong D:\hobby_project\cocopila\r2AI_2026\rag_module\ViFinQA\processed_data


Reading CSV metadata: 100%|██████████| 163128/163128 [43:05<00:00, 63.09it/s] 

Đã nạp metadata từ 163128 đường dẫn CSV.


In [ ]:
# Kết nối Qdrant và Cập nhật Payload với tên bảng mới
qdrant = QdrantClient(path=str(QDRANT_DB_PATH))
all_points = []
offset = None
while True:
    res, next_offset = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        limit=500,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    all_points.extend(res)
    if next_offset is None:
        break
    offset = next_offset

print(f"Tổng số Qdrant points: {len(all_points)}")

updated_qdrant_count = 0
matched_qdrant_count = 0
for pt in tqdm(all_points, desc="Updating Qdrant points"):
    p = pt.payload or {}
    pt_csv = p.get("csv_path", "")
    pt_tep = p.get("Tep_Nguon", "")
    rel_path = normalize_rel_path(str(pt_csv))
    base_tep = re.sub(r"@line_\d+", "", str(pt_tep)).strip()
    meta = csv_meta_by_relpath.get(rel_path) or csv_meta_by_base_tep.get(base_tep)
    if meta:
        matched_qdrant_count += 1
        new_payload = {
            "Ten_Bang": meta["Ten_Bang"],
            "Tep_Nguon": meta["Tep_Nguon"],
            "table_line": meta["table_line"],
            "source_line": meta["source_line"],
            "csv_path": meta["csv_path"],
        }
        qdrant.set_payload(
            collection_name=COLLECTION_NAME,
            payload=new_payload,
            points=[pt.id],
        )
        updated_qdrant_count += 1
    else:
        orig_ten = str(p.get("Ten_Bang", ""))
        orig_tep = str(p.get("Tep_Nguon", ""))
        l_num = extract_line_number(orig_tep, Path(pt_csv))
        base_ten = re.sub(r"\s*@line_\d+", "", orig_ten).strip()
        base_tep_clean = re.sub(r"@line_\d+", "", orig_tep).strip()
        new_ten = f"{base_ten} @line_{l_num}" if base_ten else f"Table @line_{l_num}"
        new_tep = f"{base_tep_clean}@line_{l_num}"
        qdrant.set_payload(
            collection_name=COLLECTION_NAME,
            payload={
                "Ten_Bang": new_ten,
                "Tep_Nguon": new_tep,
                "table_line": l_num,
                "source_line": l_num,
            },
            points=[pt.id],
        )
        updated_qdrant_count += 1

print(f"Đã cập nhật {updated_qdrant_count} points trên Qdrant DB.")

C:\Users\Djuybu\AppData\Local\Temp\ipykernel_26936\2573643536.py:2: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <financial_tables> contains 451386 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant = QdrantClient(path=str(QDRANT_DB_PATH))


In [ ]:
# Đồng bộ BM25 Index
if BM25_PATH.exists():
    with open(BM25_PATH, "rb") as f:
        bm25_data = pickle.load(f)
    doc_mapping = bm25_data.get("doc_mapping", [])
    bm25_updated = 0
    for doc in doc_mapping:
        doc_csv = doc.get("csv_path", "")
        doc_tep = doc.get("Tep_Nguon", "")
        rel_path = normalize_rel_path(str(doc_csv))
        base_tep = re.sub(r"@line_\d+", "", str(doc_tep)).strip()
        meta = csv_meta_by_relpath.get(rel_path) or csv_meta_by_base_tep.get(base_tep)
        if meta:
            doc["Ten_Bang"] = meta["Ten_Bang"]
            doc["Tep_Nguon"] = meta["Tep_Nguon"]
            doc["table_line"] = meta["table_line"]
            doc["source_line"] = meta["source_line"]
            doc["csv_path"] = meta["csv_path"]
        else:
            l_num = extract_line_number(str(doc_tep), Path(doc_csv))
            base_ten = re.sub(r"\s*@line_\d+", "", str(doc.get("Ten_Bang", ""))).strip()
            base_tep_clean = re.sub(r"@line_\d+", "", str(doc_tep)).strip()
            doc["Ten_Bang"] = f"{base_ten} @line_{l_num}" if base_ten else f"Table @line_{l_num}"
            doc["Tep_Nguon"] = f"{base_tep_clean}@line_{l_num}"
            doc["table_line"] = l_num
            doc["source_line"] = l_num
        bm25_updated += 1
    with open(BM25_PATH, "wb") as f:
        pickle.dump(bm25_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Đã đồng bộ {bm25_updated} mục trong file BM25 index.")

In [ ]:
# Kiểm tra kết quả mẫu
res, _ = qdrant.scroll(collection_name=COLLECTION_NAME, limit=10)
print("=== Mẫu kết quả sau khi cập nhật Qdrant ===")
for pt in res:
    p = pt.payload
    print("Ten_Bang :", p.get("Ten_Bang"))
    print("Tep_Nguon:", p.get("Tep_Nguon"))
    print("table_line:", p.get("table_line"))
    print("-" * 50)